<a href="https://colab.research.google.com/github/IshratMubarak/C-language-Code-and-Projects/blob/main/HousePricePrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# HOUSE PRICE PREDICTION: MULTI-MODEL COMPARISON (GOOGLE COLAB)
# ==========================================================

# ----------------------------------------------------------
# 1. Install Required Libraries
# ----------------------------------------------------------
!pip -q install pandas numpy matplotlib seaborn scikit-learn joblib

# ----------------------------------------------------------
# 2. Import Libraries
# ----------------------------------------------------------
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score

# ----------------------------------------------------------
# 3. Upload Dataset
# ----------------------------------------------------------
print("Please upload your HousePricePrediction CSV file")
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename)
print("\nDataset Loaded Successfully!")

# ----------------------------------------------------------
# 4. Target-Focused & Global Feature Outlier Removal
# ----------------------------------------------------------
if "Id" in df.columns:
    df.drop(columns=["Id"], inplace=True)

# Drop rows where SalePrice is missing
df = df.dropna(subset=["SalePrice"])
print(f"\nOriginal dataset shape: {df.shape}")

# List of key numerical columns to clean of outliers dynamically if they exist
outlier_cols = ["SalePrice", "GrLivArea", "LotArea", "TotalBsmtSF"]
for col in outlier_cols:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

print(f"Dataset shape after robust outlier removal: {df.shape}")

X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

# ----------------------------------------------------------
# 5. Train/Test Split
# ----------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# ----------------------------------------------------------
# 6. Preprocessing Pipeline (Crucial for Linear & SVM models!)
# ----------------------------------------------------------
# Linear Models and SVMs are highly sensitive to feature scales.
# We include StandardScaler for numerical inputs.
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

# ----------------------------------------------------------
# 7. Define Models (Adding Linear Regression & SVM)
# ----------------------------------------------------------
models = {
    "Linear Regression": LinearRegression(),

    # We restrict max_depth and min_samples_leaf to actively fight overfitting
    "Random Forest (Tuned)": RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42
    ),

    # SVR with a Radial Basis Function (RBF) kernel works well for non-linear housing data
    "Support Vector Machine (SVR)": SVR(kernel="rbf", C=100000, epsilon=0.1)
}

# ----------------------------------------------------------
# 8. Train and Evaluate All Models
# ----------------------------------------------------------
results = {}
best_r2 = -float("inf")
best_model_name = ""
best_pipeline = None

print("\n==============================")
print("TRAINING AND EVALUATION")
print("==============================")

for name, model in models.items():
    # Build a dedicated pipeline for each model
    clf_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Train
    clf_pipeline.fit(X_train, y_train)

    # Predict
    y_pred = clf_pipeline.predict(X_test)

    # Evaluate
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results[name] = {"MAE": mae, "R2": r2, "Pipeline": clf_pipeline}
    print(f"\n--- {name} ---")
    print(f"Mean Absolute Error : ${mae:,.2f}")
    print(f"R² Score            : {r2:.4f}")

    # Track the best performing model based on R² Score
    if r2 > best_r2:
        best_r2 = r2
        best_model_name = name
        best_pipeline = clf_pipeline

print(f"\n>>> Best Model: {best_model_name} with R² = {best_r2:.4f} <<<")

# ----------------------------------------------------------
# 9. Visualize Model Performance Comparison
# ----------------------------------------------------------
model_names = list(results.keys())
r2_scores = [results[m]["R2"] for m in model_names]

plt.figure(figsize=(8, 4))
sns.barplot(x=r2_scores, y=model_names, palette="viridis")
plt.title("Model Comparison (R² Score)")
plt.xlabel("R² Score (Higher is Better)")
plt.xlim(0, 1)
plt.tight_layout()
plt.savefig("Model_Comparison.png", dpi=300)
plt.show()

# ----------------------------------------------------------
# 10. Save the Best Performing Model & Cleaned Data
# ----------------------------------------------------------
df.to_csv("Cleaned_HousePricePrediction.csv", index=False)
joblib.dump(best_pipeline, "Best_HousePrice_Model.pkl")
print(f"\nSaved the best model ({best_model_name}) to 'Best_HousePrice_Model.pkl'")

# ----------------------------------------------------------
# 11. HOUSE PRICE PREDICTION INTERFACE (Using Best Model)
# ----------------------------------------------------------
print("\n==========================================")
print("HOUSE PRICE PREDICTION INTERFACE")
print("==========================================")
choice = input("Do you want to predict a house price using the best model? (yes/no): ").strip().lower()

if choice == "yes":
    new_house = pd.DataFrame(columns=X.columns)
    for col in num_cols:
        new_house.loc[0, col] = df[col].median()
    for col in cat_cols:
        new_house.loc[0, col] = df[col].mode()[0]

    print("\nEnter values for the house. Press ENTER to keep the default value.\n")
    for col in X.columns:
        value = input(f"{col} (Default = {new_house.loc[0,col]}): ")
        if value != "":
            if col in num_cols:
                new_house.loc[0, col] = float(value)
            else:
                new_house.loc[0, col] = value

    predicted_price = best_pipeline.predict(new_house)
    print("\n===================================")
    print("PREDICTED HOUSE PRICE")
    print("===================================")
    print(f"Estimated Sale Price Using {best_model_name}: ${predicted_price[0]:,.2f}")
else:
    print("\nPrediction skipped.")

# ----------------------------------------------------------
# 12. Download Files
# ----------------------------------------------------------
print("\nDownloading files...")
files.download("Cleaned_HousePricePrediction.csv")
files.download("Best_HousePrice_Model.pkl")
print("\nProgram Completed Successfully!")

Please upload your HousePricePrediction CSV file
